# Modelo 3 — Lei de Okun: Hiato do Produto com Série FGV/IBRE
## Fase 3b da Análise Econométrica (TCC - UFRPE)

Este notebook estima a relação entre o **nível** da taxa de desemprego e o **hiato do produto**,
utilizando a série de hiato calculada pelo **FGV/IBRE** com abordagem de função de produção,
em contraste com os filtros estatísticos do Modelo 2.

| Característica | Filtros Estatísticos (Modelo 2) | FGV/IBRE (Este Modelo) |
|---|---|---|
| **Fonte** | IBGE PIB + filtros HP/Hamilton/CF | FGV/IBRE — série própria |
| **Metodologia** | Decomposição tendência-ciclo | Função de produção (PTF, K, L) |
| **Interpretação** | Ciclo estatístico | Hiato econômico estrutural |
| **Disponibilidade** | Automática (nossos dados) | Arquivo externo (xlsx) |

A metodologia FGV decompõe o PIB potencial com base em:
- **PTF** — Produtividade Total dos Fatores (resíduo de Solow)
- **K util** — Estoque de capital ajustado pela utilização da capacidade instalada  
- **L util** — Força de trabalho ajustada pela taxa de participação

> **Modelo estimado:** `u_t = β₀ + β₁·hiato_fgv_t + γ₁·D_PMENova + γ₂·D_PNADc + ε_t`  
> Estimado por OLS com correção HAC (Newey-West, maxlags=4)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.filters.hp_filter import hpfilter
from statsmodels.stats.diagnostic import het_white, acorr_breusch_godfrey
from statsmodels.stats.stattools import jarque_bera   # retorna (stat, p, skew, kurt)
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

# ── PALETA E PERIODOS DE METODOLOGIA ──────────────────────────────────────────
COR_FGV       = '#7C3AED'   # roxo — cor principal deste notebook
COR_HAMILTON  = '#059669'   # verde — Hamilton do Modelo 2 (comparacao)
COR_FITTED    = '#DC2626'
COR_ZERO      = '#6B7280'
ALPHA_SHADE   = 0.08

PERIODOS_MET = [
    ('1994-03-31', '2002-03-31', '#FBBF24', 'PME Antiga'),
    ('2002-03-31', '2012-03-31', '#86EFAC', 'PME Nova'),
    ('2012-03-31', '2025-01-01', '#93C5FD', 'PNADc'),
]

def sombrear_periodos(ax):
    for ini, fim, cor, label in PERIODOS_MET:
        ax.axvspan(pd.Timestamp(ini), pd.Timestamp(fim),
                   alpha=ALPHA_SHADE, color=cor, label=label)

# ── CARREGAMENTO DOS DADOS TRATADOS ──────────────────────────────────────────
pib_df        = pd.read_pickle('../dados/pib_tratado.pkl')
desemprego_df = pd.read_pickle('../dados/desemprego_tratado.pkl')

df = pib_df.join(desemprego_df[['desemprego']], how='inner')
df = df.rename(columns={'desemprego': 'u_t'})
df = df.loc['1996-01-01':'2024-12-31'].copy()
df = df.dropna(subset=['ln_pib', 'u_t'])

print(f"Dataset base: {len(df)} obs  |  {df.index[0].date()} -> {df.index[-1].date()}")
print(df[['u_t', 'ln_pib']].describe().round(3))

In [ ]:
# ── CARREGAMENTO E TRATAMENTO DA SERIE FGV/IBRE ───────────────────────────────
# O arquivo tem 2 linhas de cabecalho:
#   Linha 1: grupos (Indices, Indices 1983=100, Acumulado em 4 trimestres)
#   Linha 2: nomes das variaveis (PTF, K util, K pot, L util, L pot, PIB efet, PIB pot, Hiato, ...)
# Dados comecam na linha 3. Coluna 0 = periodo (ex: '1996Q1'), Coluna 8 = Hiato (decimal)

df_fgv = pd.read_excel(
    '../dados/hiato_do_pib_3t25_final_FGV.xlsx',
    sheet_name='PIB',
    header=None,
    skiprows=2,
    usecols=[0, 8],
    names=['periodo', 'hiato_fgv_dec']
)

# Remover linhas sem periodo valido
df_fgv = df_fgv.dropna(subset=['periodo']).copy()
df_fgv['hiato_fgv_dec'] = pd.to_numeric(df_fgv['hiato_fgv_dec'], errors='coerce')

# Converter formato '1996Q1' para Timestamp fim de trimestre (ex: 1996-03-31)
def qstr_to_date(s):
    s = str(s).strip()
    year = int(s[:4])
    q    = int(s[5])      # 1,2,3,4
    month = q * 3         # Q1->3, Q2->6, Q3->9, Q4->12
    return pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)

df_fgv['data'] = df_fgv['periodo'].apply(qstr_to_date)
df_fgv = df_fgv.set_index('data').sort_index()

# Converter para porcentagem do PIB potencial
df_fgv['hiato_fgv'] = df_fgv['hiato_fgv_dec'] * 100

print(f"FGV: {len(df_fgv)} obs  |  {df_fgv.index[0].date()} -> {df_fgv.index[-1].date()}")
print(f"Hiato FGV — min: {df_fgv['hiato_fgv'].min():.2f}%  max: {df_fgv['hiato_fgv'].max():.2f}%  mean: {df_fgv['hiato_fgv'].mean():.2f}%")

# Merge com o dataset principal (apenas periodo de estimacao: ate 2024-T4)
df = df.join(df_fgv[['hiato_fgv']], how='left')

n_validos = df['hiato_fgv'].dropna().shape[0]
print(f"\nApos merge — observacoes com hiato FGV valido: {n_validos}")
print(df[['u_t', 'hiato_fgv']].dropna().tail())

In [ ]:
# ── GRAFICO 13: SERIE DO HIATO FGV/IBRE ──────────────────────────────────────
serie_fgv = df['hiato_fgv'].dropna()

fig, ax = plt.subplots(figsize=(14, 5))
sombrear_periodos(ax)
ax.axhline(0, color=COR_ZERO, linewidth=0.8, linestyle='--', alpha=0.7)
ax.fill_between(serie_fgv.index, serie_fgv, 0,
                where=(serie_fgv >= 0), alpha=0.35, color=COR_FGV, label='Expansao (hiato > 0)')
ax.fill_between(serie_fgv.index, serie_fgv, 0,
                where=(serie_fgv < 0),  alpha=0.35, color='#EF4444', label='Recessao (hiato < 0)')
ax.plot(serie_fgv.index, serie_fgv, color=COR_FGV, linewidth=1.5)

# Anotar eventos macroeconomicos
eventos = [
    ('1998-12-31', 'Crise Asiatica\n/Russa'),
    ('2002-09-30', 'Crise\nEleitoral'),
    ('2008-09-30', 'Crise\nFinanceira'),
    ('2015-06-30', 'Recessao\n2015-16'),
    ('2020-03-31', 'COVID-19'),
]
for dt, txt in eventos:
    ts = pd.Timestamp(dt)
    if ts in serie_fgv.index:
        val = serie_fgv.loc[ts]
        ax.annotate(txt, xy=(ts, val),
                    xytext=(ts, val + (2.5 if val < 0 else -2.5)),
                    fontsize=7, ha='center', color='#374151',
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

ax.set_ylabel('% do PIB potencial', fontsize=10)
ax.set_title('Hiato do Produto — FGV/IBRE (Funcao de Producao)\nBrasil 1982-2025', fontsize=12, fontweight='bold')
ax.set_xlim(pd.Timestamp('1996-01-01'), pd.Timestamp('2024-12-31'))
ax.legend(fontsize=8, loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(4))

desvio = serie_fgv.loc['1996-01-01':'2024-12-31'].std()
ax.text(0.98, 0.95, f'sigma = {desvio:.2f}%', transform=ax.transAxes,
        fontsize=8, color=COR_FGV, va='top', ha='right')

plt.tight_layout()
plt.savefig('../figuras/fase3_modelo2/fig_13_hiato_fgv_serie.png', dpi=150, bbox_inches='tight')
print('fig_13 salva')
plt.show()

In [ ]:
# ── GRAFICO 14: COMPARACAO FGV vs HAMILTON ────────────────────────────────────
# Computar filtro Hamilton para comparacao visual
def hamilton_filter(series, h=8, p=4):
    s = series.copy()
    X = pd.DataFrame({'const': 1.0}, index=s.index)
    for i in range(p):
        X[f'lag{h+i}'] = s.shift(h + i)
    valid = X.dropna().index
    modelo = sm.OLS(s.loc[valid], X.loc[valid]).fit()
    fitted = pd.Series(np.nan, index=s.index)
    fitted.loc[valid] = modelo.fittedvalues
    return (s - fitted) * 100, fitted

ciclo_ham, _ = hamilton_filter(df['ln_pib'])
df['hiato_hamilton'] = ciclo_ham

# Grafico comparativo
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
fig.suptitle('Comparacao: Hiato FGV/IBRE vs Filtro Hamilton (2018)\nBrasil 1996-2024',
             fontsize=13, fontweight='bold', y=0.98)

for ax, (col, cor, titulo) in zip(axes, [
    ('hiato_fgv',      COR_FGV,      'FGV/IBRE — Funcao de Producao (PTF + K + L)'),
    ('hiato_hamilton', COR_HAMILTON, 'Hamilton (2018) — Filtro Estatistico (h=8, p=4)'),
]):
    sombrear_periodos(ax)
    serie = df[col].dropna()
    ax.axhline(0, color=COR_ZERO, linewidth=0.8, linestyle='--', alpha=0.7)
    ax.fill_between(serie.index, serie, 0,
                    where=(serie >= 0), alpha=0.3, color=cor)
    ax.fill_between(serie.index, serie, 0,
                    where=(serie < 0), alpha=0.3, color='#EF4444')
    ax.plot(serie.index, serie, color=cor, linewidth=1.4)
    ax.set_ylabel('% do PIB potencial', fontsize=9)
    ax.set_title(titulo, fontsize=10, fontweight='bold')
    sigma = serie.std()
    ax.text(0.01, 0.94, f'sigma={sigma:.2f}%', transform=ax.transAxes, fontsize=8, color=cor, va='top')

# Correlacao
comum = df[['hiato_fgv', 'hiato_hamilton']].dropna()
corr_fgv_ham = comum.corr().iloc[0, 1]
axes[0].text(0.99, 0.94, f'Correlacao FGV x Hamilton: {corr_fgv_ham:.3f}',
             transform=axes[0].transAxes, fontsize=8, ha='right', va='top',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.8))

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[-1].xaxis.set_major_locator(mdates.YearLocator(4))

plt.tight_layout()
plt.savefig('../figuras/fase3_modelo2/fig_14_comparacao_fgv_hamilton.png', dpi=150, bbox_inches='tight')
print('fig_14 salva')
print(f'Correlacao FGV x Hamilton: {corr_fgv_ham:.3f}')
plt.show()

In [ ]:
# ── STEP DUMMIES PARA O MODELO DE NIVEL ───────────────────────────────────────
# Quebras metodologicas criam shifts permanentes em u_t => step dummies (0->1)
df['d_PMENova'] = (df.index >= '2002-03-31').astype(float)
df['d_PNADc']   = (df.index >= '2012-03-31').astype(float)

print('Step dummies criadas:')
print(f'  d_PMENova = 1 a partir de 2002-T1 ({df["d_PMENova"].sum():.0f} obs = 1)')
print(f'  d_PNADc   = 1 a partir de 2012-T1 ({df["d_PNADc"].sum():.0f} obs = 1)')

# Dataset de estimacao (apenas obs com hiato_fgv valido)
df_est = df[['u_t', 'hiato_fgv', 'd_PMENova', 'd_PNADc']].dropna().copy()
print(f'\nAmostra de estimacao: {len(df_est)} obs  |  {df_est.index[0].date()} -> {df_est.index[-1].date()}')

In [ ]:
# ── ESTIMACAO: OLS com HAC ────────────────────────────────────────────────────
# u_t = beta0 + beta1*hiato_fgv + gamma1*d_PMENova + gamma2*d_PNADc + eps

Y = df_est['u_t']
X = sm.add_constant(df_est[['hiato_fgv', 'd_PMENova', 'd_PNADc']])
modelo_fgv = sm.OLS(Y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})

print(modelo_fgv.summary())

# Testes de diagnostico
resid = modelo_fgv.resid
jb_stat, jb_p, jb_sk, jb_ku = jarque_bera(resid)
bg_stat, bg_p, _, _           = acorr_breusch_godfrey(modelo_fgv, nlags=4)
white_stat, white_p, _, _     = het_white(resid, modelo_fgv.model.exog)

print(f'\nTestes de diagnostico:')
print(f'  Jarque-Bera  p = {jb_p:.4f}  {"Normal" if jb_p > 0.05 else "Nao-Normal"}')
print(f'  Breusch-Godfrey p = {bg_p:.4f}  (HAC corrige autocorrelacao)')
print(f'  White        p = {white_p:.4f}  {"Homocedastico" if white_p > 0.05 else "Heterocedastico"}')

In [ ]:
# ── GRAFICO 15: REAL vs ESTIMADO ──────────────────────────────────────────────
beta0  = modelo_fgv.params['const']
beta1  = modelo_fgv.params['hiato_fgv']
gamma1 = modelo_fgv.params['d_PMENova']
gamma2 = modelo_fgv.params['d_PNADc']
r2_adj = modelo_fgv.rsquared_adj
aic    = modelo_fgv.aic

fig, ax = plt.subplots(figsize=(14, 5))
sombrear_periodos(ax)
ax.plot(df_est.index, df_est['u_t'],
        color='#2563EB', linewidth=1.5, label='Desemprego real (u_t)', alpha=0.9)
ax.plot(df_est.index, modelo_fgv.fittedvalues,
        color=COR_FGV, linewidth=1.8, linestyle='--',
        label=f'Estimado — Hiato FGV  (beta1={beta1:.4f}, p={modelo_fgv.pvalues["hiato_fgv"]:.4f})')

ax.set_ylabel('Taxa de Desemprego (%)', fontsize=10)
ax.set_title('Modelo de Hiato FGV/IBRE — Real vs. Estimado', fontsize=12, fontweight='bold')
ax.annotate(
    f'R2 Aj = {r2_adj:.3f}\nAIC = {aic:.1f}\nbeta1 = {beta1:.4f}',
    xy=(0.02, 0.92), xycoords='axes fraction', fontsize=9,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.8)
)
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(4))

plt.tight_layout()
plt.savefig('../figuras/fase3_modelo2/fig_15_modelo_hiato_fgv.png', dpi=150, bbox_inches='tight')
print('fig_15 salva')
plt.show()

In [ ]:
# ── GRAFICO 16: DIAGNOSTICO DE RESIDUOS ───────────────────────────────────────
fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

ax1 = fig.add_subplot(gs[0])
xmin, xmax = resid.min(), resid.max()
x_vals = np.linspace(xmin * 1.3, xmax * 1.3, 200)
ax1.hist(resid, bins=20, density=True, color=COR_FGV, alpha=0.7,
         edgecolor='white', linewidth=0.5)
mu, sigma = resid.mean(), resid.std()
ax1.plot(x_vals, stats.norm.pdf(x_vals, mu, sigma), color=COR_FITTED, linewidth=2,
         label=f'N({mu:.3f}, {sigma:.3f})')
jb_label = 'Normal' if jb_p > 0.05 else 'Nao-Normal'
ax1.set_title(f'Distribuicao dos Residuos\nJarque-Bera p={jb_p:.3f} ({jb_label})',
              fontsize=10, fontweight='bold')
ax1.set_xlabel('Residuo')
ax1.set_ylabel('Densidade')
ax1.legend(fontsize=8)
ax1.text(0.97, 0.95, f'Skew: {jb_sk:.2f}\nKurtosis: {jb_ku:.2f}',
         transform=ax1.transAxes, ha='right', va='top', fontsize=8)

ax2 = fig.add_subplot(gs[1])
(osm, osr), (slope, intercept, r) = stats.probplot(resid, dist='norm')
ax2.scatter(osm, osr, color=COR_FGV, alpha=0.7, s=20, zorder=3)
x_line = np.array([min(osm), max(osm)])
ax2.plot(x_line, slope * x_line + intercept, color=COR_FITTED, linewidth=1.5,
         label=f'Linha teorica (r={r:.3f})')
ax2.set_title('Q-Q Plot Normal', fontsize=10, fontweight='bold')
ax2.set_xlabel('Quantis Teoricos N(0,1)')
ax2.set_ylabel('Quantis Observados')
ax2.legend(fontsize=8)

fig.suptitle('Diagnostico de Residuos — Modelo Hiato FGV/IBRE',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figuras/fase3_modelo2/fig_16_residuos_fgv.png', dpi=150, bbox_inches='tight')
print('fig_16 salva')
plt.show()

In [ ]:
# ── RESULTADOS FINAIS E NAIRU IMPLICITO ───────────────────────────────────────
print('=' * 65)
print('COEFICIENTE DE OKUN — MODELO FGV/IBRE (NIVEL)')
print('=' * 65)
print(f'  beta0  (intercepto):      {beta0:.4f}  (p={modelo_fgv.pvalues["const"]:.4f})')
print(f'  beta1  (hiato -> desempr): {beta1:.4f}  (p={modelo_fgv.pvalues["hiato_fgv"]:.4f})')
print(f'  gamma1 (step PME Nova):   {gamma1:.4f}  (p={modelo_fgv.pvalues["d_PMENova"]:.4f})')
print(f'  gamma2 (step PNADc):      {gamma2:.4f}  (p={modelo_fgv.pvalues["d_PNADc"]:.4f})')
print(f'  R2 Ajustado: {r2_adj:.3f}  |  AIC: {aic:.1f}  |  BIC: {modelo_fgv.bic:.1f}')
print(f'  JB p={jb_p:.3f} {"Normal" if jb_p > 0.05 else "Nao-Normal"}  |  White p={white_p:.3f} {"Homocedastico" if white_p > 0.05 else "Heterocedastico"}')

# NAIRU implicito: taxa de desemprego quando hiato = 0
# Tres periodos metodologicos
nairu_pme_antiga = beta0
nairu_pme_nova   = beta0 + gamma1
nairu_pnadc      = beta0 + gamma1 + gamma2

print()
print('=' * 65)
print('NAIRU IMPLICITO (u_t quando hiato_fgv = 0)')
print('=' * 65)
print(f'  PME Antiga (pre-2002-T1):  {nairu_pme_antiga:.2f}%')
print(f'  PME Nova   (2002-T1 a 2012-T1): {nairu_pme_nova:.2f}%')
print(f'  PNADc      (pos-2012-T1):  {nairu_pnadc:.2f}%')
print()
print('Nota: A literatura internacional frequentemente cita ~4-5% como')
print('desemprego "natural" (EUA). Para o Brasil, o NAIRU estrutural e')
print('historicamente mais elevado: BCB/FGV estimam 8-12% dependendo do')
print('periodo. O valor acima reflete a NAIRU implicita no modelo de nivel.')

print()
print('INTERPRETACAO DO COEFICIENTE DE OKUN (beta1):')
print(f'  Para cada 1 p.p. de hiato negativo (PIB abaixo do potencial),')
print(f'  a taxa de desemprego aumenta {abs(beta1):.3f} p.p.')
if modelo_fgv.pvalues['hiato_fgv'] < 0.05:
    print('  -> Estatisticamente significativo a 5% (com HAC)')
else:
    print('  -> NAO significativo a 5% (com HAC)')

In [ ]:
# ── COMPARACAO COM MODELO 2 (HAMILTON) ───────────────────────────────────────
# Estimar Hamilton para comparacao direta (mesmo dataset)
df_ham = df[['u_t', 'hiato_hamilton', 'd_PMENova', 'd_PNADc']].dropna().copy()
Y_h = df_ham['u_t']
X_h = sm.add_constant(df_ham[['hiato_hamilton', 'd_PMENova', 'd_PNADc']])
modelo_ham = sm.OLS(Y_h, X_h).fit(cov_type='HAC', cov_kwds={'maxlags': 4})

jb_s_h, jb_p_h, _, _ = jarque_bera(modelo_ham.resid)
_, white_p_h, _, _    = het_white(modelo_ham.resid, modelo_ham.model.exog)

print('\n' + '=' * 70)
print('TABELA COMPARATIVA: HAMILTON (Modelo 2) vs FGV/IBRE (Modelo 3)')
print('=' * 70)
print(f'{"Criterio":<28} {"Hamilton":>12} {"FGV/IBRE":>12}')
print('-' * 70)
print(f'{"Observacoes":<28} {len(df_ham):>12} {len(df_est):>12}')
print(f'{"beta1 (coef. Okun)":<28} {modelo_ham.params["hiato_hamilton"]:>12.4f} {beta1:>12.4f}')
print(f'{"p-valor beta1":<28} {modelo_ham.pvalues["hiato_hamilton"]:>12.4f} {modelo_fgv.pvalues["hiato_fgv"]:>12.4f}')
print(f'{"R2 Ajustado":<28} {modelo_ham.rsquared_adj:>12.3f} {r2_adj:>12.3f}')
print(f'{"AIC":<28} {modelo_ham.aic:>12.1f} {aic:>12.1f}')
print(f'{"BIC":<28} {modelo_ham.bic:>12.1f} {modelo_fgv.bic:>12.1f}')
print(f'{"JB p-valor":<28} {jb_p_h:>12.3f} {jb_p:>12.3f}')
print(f'{"White p-valor":<28} {white_p_h:>12.3f} {white_p:>12.3f}')
print()
melhor = 'Hamilton' if modelo_ham.aic < aic else 'FGV/IBRE'
delta_aic = abs(modelo_ham.aic - aic)
print(f'Melhor modelo por AIC: {melhor} (delta AIC = {delta_aic:.1f})')